In [2]:
import numpy as np
import pandas as pd

In [3]:
df=pd.read_csv('/content/all_exoplanets_2021.csv.crdownload')

In [4]:
df.size

105225

In [5]:
df.isnull().sum()

,0
No.,0
Planet Name,0
Planet Host,0
Num Stars,0
Num Planets,0
Discovery Method,0
Discovery Year,0
Discovery Facility,0
Orbital Period Days,162
Orbit Semi-Major Axis,1812


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4575 entries, 0 to 4574
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   No.                            4575 non-null   int64  
 1   Planet Name                    4575 non-null   object 
 2   Planet Host                    4575 non-null   object 
 3   Num Stars                      4575 non-null   int64  
 4   Num Planets                    4575 non-null   int64  
 5   Discovery Method               4575 non-null   object 
 6   Discovery Year                 4575 non-null   int64  
 7   Discovery Facility             4575 non-null   object 
 8   Orbital Period Days            4413 non-null   float64
 9   Orbit Semi-Major Axis          2763 non-null   float64
 10  Mass                           2006 non-null   float64
 11  Eccentricity                   1707 non-null   float64
 12  Insolation Flux                370 non-null    f

In [7]:
df = df.drop(['No.', 'Planet Name', 'Planet Host',
              'Discovery Method', 'Discovery Facility',
              'Spectral Type', 'Stellar Metallicity Ratio'], axis=1, errors='ignore')

Handling missing values(i used median)

In [8]:
num_cols = df.select_dtypes(include=['float64','int64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())


In [9]:
def habitability_label(row):
    score = 0

    # Rule 1: Distance
    if 0.95 <= row['Orbit Semi-Major Axis'] <= 1.37:
        score += 1

    # Rule 2: Temperature
    if 250 <= row['Equilibrium Temperature'] <= 330:
        score += 1

    # Rule 3: Mass
    if 0.1 <= row['Mass'] <= 5:
        score += 1

    return 1 if score >= 2 else 0


df['Habitability'] = df.apply(habitability_label, axis=1)


In [10]:
df.head(5)

,Num Stars,Num Planets,Discovery Year,Orbital Period Days,Orbit Semi-Major Axis,Mass,Eccentricity,Insolation Flux,Equilibrium Temperature,Stellar Effective Temperature,Stellar Radius,Stellar Mass,Stellar Metallicity,Stellar Surface Gravity,Distance,Gaia Magnitude,Habitability
0,2,1,2007,326.03000,1.29,6165.6000,0.231,45.7,961.0,4742.0,19.00,2.70,-0.35,2.31,93.1846,4.44038,0
1,1,1,2009,516.21997,1.53,4684.8142,0.080,45.7,961.0,4213.0,29.79,2.78,-0.02,1.93,125.3210,4.56216,0
2,1,1,2008,185.84000,0.83,1525.5000,0.000,45.7,961.0,4813.0,11.00,2.20,-0.24,2.63,75.4392,4.91781,0
3,1,2,2002,1773.40002,2.93,1481.0878,0.370,45.7,961.0,5338.0,0.93,0.90,0.41,4.45,17.9323,6.38300,0
4,3,1,1996,798.50000,1.66,565.7374,0.680,45.7,961.0,5750.0,1.13,1.08,0.06,4.36,21.1397,6.06428,0


In [11]:
df.sample(5)

,Num Stars,Num Planets,Discovery Year,Orbital Period Days,Orbit Semi-Major Axis,Mass,Eccentricity,Insolation Flux,Equilibrium Temperature,Stellar Effective Temperature,Stellar Radius,Stellar Mass,Stellar Metallicity,Stellar Surface Gravity,Distance,Gaia Magnitude,Habitability
1306,1,1,2018,11.168454,0.10356,126.49634,0.258,45.7,1030.0,6154.0,1.16,1.19,0.100,4.38,130.7200,9.72626,0
776,2,1,2011,1319.000000,2.38000,4131.62000,0.400,45.7,961.0,5966.0,1.27,1.02,-0.135,4.35,55.4847,7.63573,0
1359,3,1,2016,6.771315,0.06702,30.90000,0.251,116.0,902.0,5248.0,0.89,0.87,0.130,4.48,249.6450,12.45280,0
367,1,1,2017,4.722812,0.05025,870.85420,0.079,45.7,858.0,4803.0,0.69,0.76,0.000,4.64,229.5180,13.12250,0
3021,1,2,2014,15.955387,0.11600,220.88734,0.092,45.7,961.0,5117.0,0.72,0.96,0.020,4.64,717.5270,15.26340,0


In [12]:
df['Habitability'].value_counts()

,count
Habitability,
0,4567
1,8


In [ ]:
X = df.drop('Habitability', axis=1)
y = df['Habitability']


Apply SMOTE + Tomek Links

In [15]:
from imblearn.combine import SMOTETomek

X = df.drop('Habitability', axis=1)
y = df['Habitability']

smote_tomek = SMOTETomek()

X_resampled, y_resampled = smote_tomek.fit_resample(X, y)

print("After balancing:")
print(y_resampled.value_counts())

After balancing:
Habitability
0    4563
1    4563
Name: count, dtype: int64


SMOTE only

In [16]:
from imblearn.over_sampling import SMOTE

sm = SMOTE()
X_res, y_res = sm.fit_resample(X, y)
print(y_res.value_counts())


Habitability
0    4567
1    4567
Name: count, dtype: int64


Random Undersampling- reduce the majority class(but we lose imp data)

In [17]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler()
X_res, y_res = rus.fit_resample(X, y)
print(y_res.value_counts())


Habitability
0    8
1    8
Name: count, dtype: int64


Class Weights (no balancing required)

In [18]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(class_weight='balanced')
model.fit(X, y)


RandomForestClassifier(class_weight='balanced')

Anomaly Detection (best when minority < 1%)

In [19]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.002)  # ~8/4567
iso.fit(X)

anomaly_scores = iso.predict(X)
